In [0]:
%sql

-- Question 4: Do traffic advisories correlate with changes in pickup volume or trip duration within impacted boroughs?
-- Suggested chart: Grouped bar | X: borough | Y: avg_duration_minutes | Color/Group: severity

WITH trip_advisory AS (
    SELECT 
        t.trip_id,
        z.borough,
        t.trip_duration_min,
        t.trip_distance,
        t.total_amount,
        a.advisory_type,
        a.severity,
        ROW_NUMBER() OVER (
            PARTITION BY t.trip_id 
            ORDER BY a.severity DESC, a.effective_from DESC
        ) AS advisory_rank
    FROM nyc_mobility.mart.fact_trip AS t
    JOIN nyc_mobility.mart.dim_zone AS z
        ON t.pu_location_id = z.location_id
    LEFT JOIN nyc_mobility.mart.dim_advisory AS a
        ON LOWER(TRIM(a.borough)) = LOWER(TRIM(z.borough))
       AND t.pickup_datetime BETWEEN a.effective_from AND a.effective_to
)
SELECT 
    borough,
    COALESCE(severity, 'None') AS severity,
    COUNT(trip_id) AS total_trips,
    ROUND(AVG(trip_duration_min), 2) AS avg_duration_minutes,
    ROUND(AVG(total_amount), 2) AS avg_total_fare
FROM trip_advisory
WHERE advisory_rank = 1
GROUP BY borough, severity
ORDER BY borough, severity;